### The data
There are three data scources used to construct this project, scraped from Sofascore and WhoScored.  
- event-data-25-26 which contains event level data for all matches in the 2025/26 season. 
- player_stats_25/26 which contains detailed player level metrics for all players appearing in the Premier League across the 2025/26 season.
- team_stats_25/26 which contains detailed team level metrics for the 2025/26 season.
- player_details which contains basic player information for all players appearing in the Premier league across the 2025/26 season. This extra data sheet was needed for its detailed player position information.

Before proceeding with the analysis we will clean the data to create one player level data sheet with everything needed. The main challenge of this will be to itterate through the event data to identify and assign all SCA.


In [1]:
from pathlib import Path
import pandas as pd

project_folder = Path.cwd().parent 
raw_data_folder = project_folder / "data" / "raw" 

# Import the four data sources
event_data = pd.read_csv(raw_data_folder / "event-data-25-26.csv")
player_stats = pd.read_csv(raw_data_folder / "player_data.csv")
team_stats = pd.read_csv(raw_data_folder / "team_stats.csv")
player_details = pd.read_csv(raw_data_folder / "player_details.csv")

In [45]:
# 1: Check for duplicate player entries in the data
if not (player_stats["player id"].duplicated().any()): 
    print("No duplicate IDs in player_stats") 
if not (player_details["id"].duplicated().any()):
    print("No duplicate IDs in player_details")

# Check that the player id in player_stats matches the id in player_details
if (player_stats["player id"].isin(player_details["id"]).all()):
    print("All IDs in player_stats have a corresponding ID in player_details")

# Check the number of unique ids in both data sets
print("Players in player_stats: ", player_stats["player id"].nunique())
print("Players in player_details: ", player_details["id"].nunique())

No duplicate IDs in player_stats
No duplicate IDs in player_details
All IDs in player_stats have a corresponding ID in player_details
Players in player_stats:  537
Players in player_details:  537


In [47]:
# 2: Merge player_stats and player_details on player id 
players_df = player_stats.merge(player_details, 
                                how = "left",
                                left_on = "player id",
                                right_on = "id")
# Check it merged correctly
print(players_df[["player","player id","name", "id"]].head())

            player  player id             name       id
0  Bruno Fernandes     288205  Bruno Fernandes   288205
1        Max Weiss    1128801        Max Weiss  1128801
2      Declan Rice     856714      Declan Rice   856714
3  Bruno Guimarães     866469  Bruno Guimarães   866469
4            Rodri     827606            Rodri   827606


In [49]:
# 3: Add the team posession stats to players_df
# Check every player's team ID exists in team_stats
if (players_df["team id"].isin(team_stats["teamId"]).all()):
    print("All teams are accounted for")

# Separate team posession
team_possession = team_stats[["teamId","averageBallPossession"]]

# Round the average possession figures
team_possession["averageBallPossession"] = team_possession["averageBallPossession"].round(2)

# Merge team possession into players_df
players_df = players_df.merge(team_possession,
                             how = "left",
                             left_on = "team id",
                             right_on = "teamId")

# Check it merged correctly
print(players_df[["team id","averageBallPossession"]].head())

All teams are accounted for
   team id  averageBallPossession
0       35                  51.79
1        6                  42.29
2       42                  56.13
3       39                  52.61
4       17                  60.50


In [51]:
# 4: Handle positions, dropping GKs and CBs
# Drop the single row without a detailed position (a Wolves u-21 player)
players_df = players_df.dropna(subset = ["positions_detailed"]).copy()

# Check the data type
print(players_df["positions_detailed"].apply(type).value_counts())

# Modify to a list
import ast
players_df["positions_detailed"] = players_df["positions_detailed"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
print(players_df["positions_detailed"].apply(type).value_counts())

# Find the detailed primary position
players_df["primaryPosition"] = players_df["positions_detailed"].apply(lambda x: x[0])
# Define a position map to group players by position
position_map = {
    "ST": "Striker",
    "LW": "Winger",
    "RW": "Winger",
    "AM": "Midfielder",
    "CM": "Midfielder",
    "DM": "Midfielder",
    "MC": "Midfielder",
    "DL": "Fullback",
    "DR": "Fullback",
    "MR": "Fullback",
    "ML": "Fullback",
    "DC": "Centerback"}

players_df["positionGroup"] = players_df["primaryPosition"].map(position_map)

# Drop goalkeeprs
players_df = players_df[players_df["primaryPosition"] != "GK"].copy()

positions_detailed
<class 'str'>    536
Name: count, dtype: int64
positions_detailed
<class 'list'>    536
Name: count, dtype: int64


In [53]:
# 5: Derive unsuccessful dribbles 

import numpy as np
# Calculate the total dribbles 
players_df["totalDribbles"] = np.where(
    players_df["successfulDribblesPercentage"] > 0,
    players_df["successfulDribbles"] / (players_df["successfulDribblesPercentage"] / 100),
    0)

# Derive the unsuccessful dribbles and ensure integer values
players_df["unsuccessfulDribbles"] = (
    players_df["totalDribbles"] - players_df["successfulDribbles"]).round().astype("Int64")

In [59]:
# 6: Create a final dataframe with only essential metrics and export it
# Define the required columns
feature_cols = ["name","id","team","team id","primaryPosition","positionGroup","minutesPlayed","averageBallPossession","totalShots",
             "inaccuratePasses", "dispossessed","unsuccessfulDribbles","keyPasses","assists","expectedAssists","goals","expectedGoals",
             "market_value"]

# Create the final df
players_cleaned_df = players_df[feature_cols].copy()

# Rename columns for clarity
players_cleaned_df = players_cleaned_df.rename(columns = {
    "name" : "playerName",
    "id" : "playerId",
    "team" : "teamName",
    "team id" : "teamId",
    "market_value" : "marketValue"
})

# Fill nan values with zero
players_cleaned_df["expectedAssists"] = players_cleaned_df["expectedAssists"].fillna(0)
players_cleaned_df["expectedGoals"] = players_cleaned_df["expectedGoals"].fillna(0)
players_cleaned_df["marketValue"] = players_cleaned_df["marketValue"].fillna(0)

# final checks
print(players_cleaned_df.shape)
print(players_cleaned_df["playerId"].duplicated().sum())
print(players_cleaned_df.isna().sum())

# Export the final df as a csv file
players_cleaned_df.to_csv(project_folder / "data" / "cleaned" / "player_stats_cleaned.csv", index = False)

(496, 18)
0
playerName               0
playerId                 0
teamName                 0
teamId                   0
primaryPosition          0
positionGroup            0
minutesPlayed            0
averageBallPossession    0
totalShots               0
inaccuratePasses         0
dispossessed             0
unsuccessfulDribbles     0
keyPasses                0
assists                  0
expectedAssists          0
goals                    0
expectedGoals            0
marketValue              0
dtype: int64


In [61]:
# 7: Clean team_stats
# Define the required columns
team_cols = ["teamName", "teamId", "goalsScored","averageBallPossession"]

# Create the cleaned data frame
teams_cleaned_df = team_stats[team_cols].copy()

# Round the possession column
teams_cleaned_df["averageBallPossession"] = teams_cleaned_df["averageBallPossession"].round(2)

# Final checks
print(teams_cleaned_df.shape)
print(teams_cleaned_df["teamId"].duplicated().sum())
print(teams_cleaned_df.isna().sum())

# Export the cleaned data frame
teams_cleaned_df.to_csv(project_folder / "data" / "cleaned" / "team_stats_cleaned.csv", index = False)

(20, 4)
0
teamName                 0
teamId                   0
goalsScored              0
averageBallPossession    0
dtype: int64


### Shot Creating Actions
The main challenge of this section is identifying the SCA. We have all event data over the 25/26 season with over 577,000 entries. Before beginning extracting SCA from this data we must first define what a SCA actually is. OPTA defines it as the two offesnive actions directly proceeding a shot. Therfore, for each shot two players can be credited with a SCA, whether it is a driblle, tackle, pass or anything else. Once the SCAs have been identified, we want to extract this data to finish with a dataframe containing all players in the league and their SCA numbers over the whole season. 

In [14]:
# Understanding the data
print(event_data["type"].value_counts())
print(event_data.columns.to_list())

type
Pass               362739
BallRecovery        30575
BallTouch           26492
Aerial              24008
Clearance           21134
Foul                16452
TakeOn              13511
Tackle              12670
CornerAwarded        7598
Dispossessed         6575
Interception         6289
BlockedPass          5626
Challenge            5213
SavedShot            4978
Save                 4899
KeeperPickup         4424
MissedShots          3309
SubstitutionOff      3132
SubstitutionOn       3132
End                  2280
Card                 1532
Start                1520
OffsideGiven         1175
OffsidePass          1175
OffsideProvoked      1175
FormationChange      1129
Goal                 1045
FormationSet          760
Claim                 668
Error                 650
ShieldBallOpp         540
Punch                 494
KeeperSweeper         479
ShotOnPost            211
Smother                98
PenaltyFaced           92
GoodSkill              68
ChanceMissed           23
CrossNo

In [25]:
# Shot creating action identification algortithm
def calculate_sca(match):

    # Define events that can qualify as a SCA
    SCA_TYPES = {
        "Pass",
        "BallTouch",
        "Foul",
        "TakeOn",
        "CornerAwarded",
        "SavedShot",
        "ShotOnPost",
        "GoodSkill"}
    
    sca_results = []
    
    shots = match[match["is_shot"] == True]
    
    for shot_idx in shots.index:
        
        shot = match.loc[shot_idx]
    
        scas = []
    
        i = shot_idx - 1
    
        while i >= 0:
               
            event = match.loc[i]
            
            # Opposition Event
            if event["team_id"] != shot["team_id"]:
    
                # Successful opposition event - possession ended
                if event["outcome_type"] == "Successful":
                    break
                # Unsuccessful opposition event - Ignore and continue
                i -= 1
                continue 
    
            # Same-team event - possession continues
            if event["outcome_type"] != "Successful":
                i -= 1
                continue
    
            # Check if event qualifies as a SCA
            if event["type"] not in SCA_TYPES:
                i -= 1
                continue
    
            
            # Don't count same player for two SCAs
            if any(sca["player_id"] == event["player_id"] for sca in scas):
                i -= 1
                continue
    
            # Log the SCA
            scas.append({
                "player_id": event["player_id"],
                "player": event["player"],
                "type": event["type"],
                "row": i})
            
            # Special events which cannot have secondary SCA  
            if event["type"] in {"Foul", "SavedShot", "ShotOnPost"}:
                break
    
            # Stop after 2 SCAs
            if len(scas) == 2:
                break
            
            i -= 1
    
        # Store the results
        result = {
        "game_id": shot["game_id"],
        "shot_player": shot["player"],
        "shot_player_id": shot["player_id"],
        "shot_type": shot["type"],
        "shot_row": shot_idx,
        
        "direct_sca_player": None,
        "direct_sca_player_id": None,
        "direct_sca_type": None,
        
        "secondary_sca_player": None,
        "secondary_sca_player_id": None,
        "secondary_sca_type": None}
        
        if len(scas) >= 1:
            result["direct_sca_player"] = scas[0]["player"]
            result["direct_sca_player_id"] = scas[0]["player_id"]
            result["direct_sca_type"] = scas[0]["type"]
        
        if len(scas) >= 2:
            result["secondary_sca_player"] = scas[1]["player"]
            result["secondary_sca_player_id"] = scas[1]["player_id"]
            result["secondary_sca_type"] = scas[1]["type"]
        
        sca_results.append(result)    

    return pd.DataFrame(sca_results)


In [41]:
# Run the function for all matches in the event data

# Find unique matches
game_ids = event_data["game_id"].unique()

# Define an empty list to store sca results
all_sca_results = []

# Calculate sca for all games
for game_id in game_ids:
    match = event_data[event_data["game_id"] == game_id].reset_index(drop = True)

    match_sca = calculate_sca(match)

    all_sca_results.append(match_sca)

# Merge into one df
all_sca_results_df = pd.concat(
    all_sca_results,
    ignore_index=True
)

# Validation
print("Total shots:", len(all_sca_results_df))
print("Games:", all_sca_results_df["game_id"].nunique())

total_shots = event_data["is_shot"].sum()

print("Shots in event data:", total_shots)
print("SCA result rows:", len(all_sca_results_df))

Total shots: 9543
Games: 380
Shots in event data: 9543
SCA result rows: 9543


In [43]:
# define all SCA types
sca_types = [
    "Pass",
    "BallTouch",
    "Foul",
    "TakeOn",
    "CornerAwarded",
    "SavedShot",
    "ShotOnPost",
    "GoodSkill"
]

# Count direct SCA
direct_sca = (
    all_sca_results_df
    .dropna(subset=["direct_sca_player"])
    .groupby(["direct_sca_player", "direct_sca_type"])
    .size()
    .unstack(fill_value=0)
)
# Make sure every SCA type exists even if it did not occur in game
direct_sca = direct_sca.reindex(columns = sca_types, fill_value = 0)

# Standardise column names
direct_sca.columns = [
    f"direct_{sca_type.lower()}_sca"
    for sca_type in direct_sca.columns
]

# Count secondary SCA
secondary_sca = (
    all_sca_results_df
    .dropna(subset=["secondary_sca_player"])
    .groupby(["secondary_sca_player", "secondary_sca_type"])
    .size()
    .unstack(fill_value = 0)
)

# Make sure every SCA type exists even if it did not occur in game  
secondary_sca = secondary_sca.reindex(columns = sca_types, fill_value = 0)

# Standardise column names
secondary_sca.columns = [
    f"secondary_{sca_type.lower()}_sca"
    for sca_type in secondary_sca.columns
]

# Merge direct and secondary SCAs
sca_df = direct_sca.join(
    secondary_sca,
    how="outer"
).fillna(0)

# Ensure count types are integer
sca_df = sca_df.astype(int)

# Rename cols
direct_cols = [
    col for col in sca_df.columns
    if col.startswith("direct_")
]
secondary_cols = [
    col for col in sca_df.columns
    if col.startswith("secondary_")
]

# Total direct SCA
sca_df["direct_sca"] = sca_df[direct_cols].sum(axis=1)

# Total secondary SCA
sca_df["secondary_sca"] = sca_df[secondary_cols].sum(axis=1)

# Final total SCA
sca_df["total_sca"] = (
    sca_df["direct_sca"] +
    sca_df["secondary_sca"]
)

# Rearrange 
sca_df = sca_df[
    [   "total_sca",
     
        "direct_sca",
        "secondary_sca",

        "direct_pass_sca",
        "direct_balltouch_sca",
        "direct_foul_sca",
        "direct_takeon_sca",
        "direct_cornerawarded_sca",
        "direct_savedshot_sca",
        "direct_shotonpost_sca",
        "direct_goodskill_sca",

        "secondary_pass_sca",
        "secondary_balltouch_sca",
        "secondary_foul_sca",
        "secondary_takeon_sca",
        "secondary_cornerawarded_sca",
        "secondary_savedshot_sca",
        "secondary_shotonpost_sca",
        "secondary_goodskill_sca"]]

sca_df = sca_df.reset_index()
sca_df = sca_df.rename(columns={"direct_sca_player": "player"})

sca_df.to_csv(project_folder / "data" / "cleaned" / "SCA.csv", index = False)